# CISA Known Exploited Vulnerabilities (KEV) Machine Learning
Data source: The Cybersecurity and Infrastructure Security Agency's (CISA) [Known Exploited Vulnerabilities (KEV) catalog](https://github.com/cisagov/kev-data?)

## Background
I previously performed an exploratory data analysis on the KEV dataset to identify trends in known exploited vulnerabilities, remediation-timelines, and ransomware-associated weaknesses. 

My findings were as follows:
- Microsoft has the largest number of cataloged vulnerabilities.
- Input validation and use-after-free are the most common weaknesses.
- Ransomware vulnerabilities show similar remediation timelines to the broader catalog.
- Most remediation deadlines are exactly 21 days.
- Vulnerability additions peaked in 2022.

The best use-case for a regression model for this project would be estimating the remediation deadline of a particular vulnerability. However, my analysis project revealed dramatic spikes in the remediation days distribution at 14, 21, and 181 days. These results suggest that CISA uses standardized deadlines. Data that can be categorized in this fashion is better suited for a classification model than a regression model. 
However, there was enough variation among vulnerabilities in general versus ransomware-associated CWEs that a classification model might be able to successfully predict ransomware status. I am more excited by the prospect of classifying ransomware status than I am by classifying remediation deadlines. Therefore, I will train my model to do the former. 

## Step 1: Import Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

## Step 2: Load the Data

In [2]:
# Get the .csv data from GitHub
data_url = "https://raw.githubusercontent.com/cisagov/kev-data/refs/heads/develop/known_exploited_vulnerabilities.csv"
kev = pd.read_csv(data_url)

# Save a local copy
kev.to_csv("known_exploited_vulnerabilities.csv", index=False)
kev = pd.read_csv("known_exploited_vulnerabilities.csv")

print(kev.head(30))

             cveID    vendorProject  \
0   CVE-2025-68686         Fortinet   
1   CVE-2026-16812           Arista   
2   CVE-2026-16232      Check Point   
3   CVE-2026-50522        Microsoft   
4   CVE-2026-60137        WordPress   
5   CVE-2026-63030        WordPress   
6    CVE-2026-0770         Langflow   
7   CVE-2021-27137           DD-WRT   
8   CVE-2026-58644        Microsoft   
9   CVE-2026-25089         Fortinet   
10  CVE-2026-39808         Fortinet   
11  CVE-2026-46817           Oracle   
12   CVE-2023-4346  KNX Association   
13  CVE-2026-56155        Microsoft   
14  CVE-2026-56164        Microsoft   
15  CVE-2026-15409        SonicWall   
16  CVE-2026-15410        SonicWall   
17   CVE-2008-4128            Cisco   
18  CVE-2026-56291          Balbooa   
19  CVE-2026-48939         iCagenda   
20  CVE-2026-48908       JoomShaper   
21  CVE-2026-55255         Langflow   
22  CVE-2026-56290         Joomlack   
23  CVE-2026-48282            Adobe   
24  CVE-2026-45659       

## Step 3: Read the Data Documentation 
### Shema
#### (Extracted from known_exploited_vulnerabilities_schema.json)

| Column | Description |
| :--- | ---: | 
| cveID | The CVE ID of the vulnerability in the format CVE-YYYY-NNNN, note that the number portion can have more than 4 digits |
| vendorProject | The vendor or project name for the vulnerability |
| product | The vulnerability product |
| vulnerabilityName | The name of the vulnerability |
| dateAdded | The date the vulnerability was added to the catalog in the format YYYY-MM-DD |
| shortDescription | A short description of the vulnerability |
| requiredAction | The required action to address the vulnerability |
| dueDate | The date the required action is due in the format YYYY-MM-DD |
| knownRansomwareCampaignUse | 'Known' if this vulnerability is known to have been leveraged as part of a ransomware campaign; 'Unknown' if CISA lacks confirmation that the vulnerability has been utilized for ransomware |
| notes | Any additional notes about the vulnerability |
| cwes | Common Weakness Enumeration (CWE) codes associated with this vulnerability. CWEs are in the format CWE-NNNN; note that the number portion can have any number of digits |


## Step 4: Inspect the Data

In [3]:
print(kev.info())
print()
print()
print(kev.describe(include="all"))
print()
print()
kev.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1655 entries, 0 to 1654
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   cveID                       1655 non-null   str  
 1   vendorProject               1655 non-null   str  
 2   product                     1655 non-null   str  
 3   vulnerabilityName           1655 non-null   str  
 4   dateAdded                   1655 non-null   str  
 5   shortDescription            1655 non-null   str  
 6   requiredAction              1655 non-null   str  
 7   dueDate                     1655 non-null   str  
 8   knownRansomwareCampaignUse  1655 non-null   str  
 9   notes                       1655 non-null   str  
 10  cwes                        1484 non-null   str  
dtypes: str(11)
memory usage: 142.4 KB
None


                 cveID vendorProject  product  \
count             1655          1655     1655   
unique            1655           276      670

np.int64(0)

## Step 5: Clean the Data
Before analysis, I cleaned the dataset by renaming columns to snake_case and converting date columns to datetime objects.

In [4]:
# Rename columns to match Python's snake_case convention
kev.rename(columns={
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use"
}, inplace=True)

#print(kev.columns)



# Covert values in date_added and due_date from strings to dates
kev["date_added"] = pd.to_datetime(kev["date_added"])
kev["due_date"] = pd.to_datetime(kev["due_date"])

#print(kev.info())

## Step 6: Choosing the Model

I already decided on a classification model for this project. Below is a table detailing my assessment of the different classification techniques I could use. 


| Technique | Is Appropriate? | Explanation |
| :--- | --- | ---: | 
| Logistic Regression | Appropriate | Excellent baseline, interpretable |
| Decision Forest (Random Forest) | Appropriate | Usually very strong on tabular data |
| Support Vector Machine (SVM) | Appropriate | Good for complex decision boundaries |
| Decision Tree | Has potential | Good for visualization, but often outperformed by forests |
| Naive Bayes | Has potential | Fast, but assumptions are often unrealistic |
| k-Nearest Neighbors | Inappropriate | Sensitive to scaling and many categorical variables |
<br>


**I will try a Logistic Regression, Decision Forest, and Support Vector Machine model--and then compare performance.**

## Step 7: Data Processing

### Importing Libraries

In [18]:
# For modifying the data to be interpretable by the models
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

### Reshaping Predictors

The predictors I am interested in are: "vendor_project", "product", and "cwes".

All of my predictors are of the str datatype. Therefore, I must either use label encoding or one-hot encoding. Label encoding is not appropriate in this case since none of my predictors have a natural order. I will use one-hot encoding via the pandas.get_dummies() method. 

#### For vendor_project values
From Step 4: Inspect the Data I know that there are 276 unique vendor_project values.
This is a lot, but it is still reasonable to one-hot encode all of the values without presorting less-common values into an "other" column.

In [6]:
kev_encoded = pd.get_dummies(kev, columns=["vendor_project"], dtype=int)

print(kev_encoded)

              cve_id                 product  \
0     CVE-2025-68686                 FortiOS   
1     CVE-2026-16812  VeloCloud Orchestrator   
2     CVE-2026-16232            SmartConsole   
3     CVE-2026-50522              SharePoint   
4     CVE-2026-60137                    Core   
...              ...                     ...   
1650  CVE-2021-27561       Device Management   
1651  CVE-2021-40539            ManageEngine   
1652  CVE-2020-10189            ManageEngine   
1653   CVE-2019-8394            ManageEngine   
1654  CVE-2020-29583       Multiple Products   

                                     vulnerability_name date_added  \
0     Fortinet FortiOS Exposure of Sensitive Informa... 2026-07-27   
1     Arista VeloCloud Orchestrator On-Prem OS Comma... 2026-07-27   
2     Check Point SmartConsole Improper Authenticati... 2026-07-22   
3     Microsoft SharePoint Deserialization of Untrus... 2026-07-22   
4            WordPress Core SQL Injection Vulnerability 2026-07-21   
...

#### For product values
From Step 4: Inspect the Data I know that there are 670 unique product values.
This is too many values to one-hot encode, so I will presort less-common values into an "Other" column.

In [7]:
product_counts = kev_encoded["product"].value_counts()
#for product, count in product_counts.items():
    #print(product, count)

# Keep only products that appear 3 or more times
common_products = product_counts[product_counts >= 3].index

kev_encoded["product"] = kev_encoded["product"].apply(
    lambda product: product if product in common_products else "Other"
)

kev_encoded = pd.get_dummies(kev_encoded, columns=["product"], dtype=int) 

print(kev_encoded)

              cve_id                                 vulnerability_name  \
0     CVE-2025-68686  Fortinet FortiOS Exposure of Sensitive Informa...   
1     CVE-2026-16812  Arista VeloCloud Orchestrator On-Prem OS Comma...   
2     CVE-2026-16232  Check Point SmartConsole Improper Authenticati...   
3     CVE-2026-50522  Microsoft SharePoint Deserialization of Untrus...   
4     CVE-2026-60137         WordPress Core SQL Injection Vulnerability   
...              ...                                                ...   
1650  CVE-2021-27561  Yealink Device Management Server-Side Request ...   
1651  CVE-2021-40539  Zoho ManageEngine ADSelfService Plus Authentic...   
1652  CVE-2020-10189  Zoho ManageEngine Desktop Central File Upload ...   
1653   CVE-2019-8394  Zoho ManageEngine ServiceDesk Plus (SDP) File ...   
1654  CVE-2020-29583  Zyxel Multiple Products Use of Hard-Coded Cred...   

     date_added                                  short_description  \
0    2026-07-27  Fortinet For

#### For cwes values
Some rows in the cwes column contain a single CWE and other rows contain a list of CWEs. I must first split these lists into invididual CWEs. Each CWE that was previously in a string will be counted as an individual. This should have a positive impact on the performance of my model.

I will need to use a Multi Label Binarizer instead of One Hot Encoding, since a single vulnerability can be associated with multiple CWEs.

In [8]:
kev_encoded["cwes"] = kev_encoded["cwes"].apply(
    lambda x: [cwe.strip() for cwe in x.split(",")] if isinstance(x, str) else []
)

#print(kev_encoded["cwes"].describe())
# There were previously 241 unique values. Now there are 242 unique values, since I replaced NA values with an empty list. 

mlb = MultiLabelBinarizer()

# Returns a numpy array
cwe_encoded = mlb.fit_transform(kev_encoded["cwes"])

# Convert the numpy array to a dataframe 
cwe_df = pd.DataFrame(
    cwe_encoded,
    columns=mlb.classes_,
    index=kev_encoded.index
)

# Remove the original cwes column and replace it with with my encoded dataframe 
kev_encoded = pd.concat(
    [kev_encoded.drop(columns=["cwes"]), cwe_df],
    axis=1
)

#print(len(mlb.classes_))
# This printed 183 unique values compared to the previous 241 unique values. This is because lists of CWEs are no longer considered unique.

print(kev_encoded.head())

           cve_id                                 vulnerability_name  \
0  CVE-2025-68686  Fortinet FortiOS Exposure of Sensitive Informa...   
1  CVE-2026-16812  Arista VeloCloud Orchestrator On-Prem OS Comma...   
2  CVE-2026-16232  Check Point SmartConsole Improper Authenticati...   
3  CVE-2026-50522  Microsoft SharePoint Deserialization of Untrus...   
4  CVE-2026-60137         WordPress Core SQL Injection Vulnerability   

  date_added                                  short_description  \
0 2026-07-27  Fortinet FortiOS contains an exposure of sensi...   
1 2026-07-27  Arista VeloCloud Orchestrator On-Prem contains...   
2 2026-07-22  Check Point SmartConsole contains an improper ...   
3 2026-07-22  Microsoft SharePoint contains a deserializatio...   
4 2026-07-21  WordPress Core contains a SQL injection vulner...   

                                     required_action   due_date  \
0  Apply mitigations in accordance with vendor in... 2026-08-10   
1  Apply mitigations in accord

In [9]:
# Dropping undesireable columns as well as the column I am trying to predict 
X = kev_encoded.drop(
    columns= [
        "cve_id",
        "vulnerability_name",
        "date_added",
        "short_description",
        "required_action",
        "due_date",
        "known_ransomware_campaign_use", #the column we are trying to predict
        "notes"
    ])

y = kev["known_ransomware_campaign_use"]

X.columns.unique

<bound method Index.unique of Index(['vendor_project_7-Zip', 'vendor_project_AMI', 'vendor_project_ASUS',
       'vendor_project_Accellion', 'vendor_project_Acclaim Systems',
       'vendor_project_Acronis', 'vendor_project_Adminer',
       'vendor_project_Adobe', 'vendor_project_Advantive',
       'vendor_project_Alcatel',
       ...
       'CWE-913', 'CWE-917', 'CWE-918', 'CWE-923', 'CWE-93', 'CWE-94',
       'CWE-940', 'CWE-95', 'CWE-96', 'CWE-98'],
      dtype='str', length=573)>

In [29]:
# Splitting the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

## Step 8: Implementing the Models

### Importing Libraries

In [27]:
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# For eveluating the models
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

### Creating a Method to Evaluate Model Performance

In [19]:
def evaluate_model(y_predict):
    print("Confusion matrix: \n" + str(confusion_matrix(y_test, y_predict)))
    # upper lefthand corner = true positives
    # lower lefthand corner = false positives
    # upper righthand corner = false negatives
    # lower righthand corner = true negatives
    
    print()
    
    print("Accuracy score: " + str(accuracy_score(y_test, y_predict)))
    print("Precision score: " + str(precision_score(y_test, y_predict, pos_label="Known")))
    print("Recall score: " + str(recall_score(y_test, y_predict, pos_label="Known")))
    print("f1 score: " + str(f1_score(y_test, y_predict, pos_label="Known")))

The Confusion Matrix evaluates how well a classification model performs by comparing predicted outcomes against true values.

The upper lefthand corner gives the count of true positives, the lower lefthand corner gives the count of false positives, the upper righthand corner gives the count of false negatives, and the lower righthand corner gives the count of true negatives.

<br>

Accuracy measures how many classifications the algorithm got correct out of every classification it made.

Precision is the ratio of correct positive classifications to all positive classifications made by the model.

Recall is the ratio of correct positive predictions classifications made by the model to all actual positives.

F1-score is a combination of precision and recall. The formula for the f1-score uses a harmonic mean, so it will be low if either precision or recall is low.


### Addressing Class Imbalance

I added the `class_weight='balanced'` argument to each model. This tells sklearn to weight the minority class ("Known", ~20% of the data) more heavily during training, so misclassifying it costs more than misclassifying the majority class.

### Logistic Regression

In [24]:
lr_model = LogisticRegression(class_weight='balanced')
lr_model.fit(X_train, y_train)

# The model predicts the y values based on the X values
y_predict = lr_model.predict(X_test)

In [25]:
evaluate_model(y_predict)

Confusion matrix: 
[[ 47  36]
 [ 80 251]]

Accuracy score: 0.7198067632850241
Precision score: 0.3700787401574803
Recall score: 0.5662650602409639
f1 score: 0.44761904761904764


### Decision Forest
Decision Trees are great in theory, but prone to overfitting in practice. A Decision Forest consists of many different Decision Trees that all work together to classify a new point. That way, even if some trees in the forest are prone to overfitting, it has less of an impact on the model as a whole.

In [14]:
rf_model = RandomForestClassifier(n_estimators = 200, random_state = 42)
rf_model.fit(X_train, y_train)

y_predict = rf_model.predict(X_test)

In [15]:
evaluate_model(y_predict)

Confusion matrix: 
[[ 22  61]
 [ 15 316]]

Accuracy score: 0.8164251207729468
Precision score: 0.5945945945945946
Recall score: 0.26506024096385544
f1 score: 0.36666666666666664


### Support Vector Machine

In [16]:
svm_model = SVC(kernel= "rbf", gamma = 1)
svm_model.fit(X_train, y_train)

y_predict = svm_model.predict(X_test)

In [17]:
evaluate_model(y_predict)

Confusion matrix: 
[[ 11  72]
 [  4 327]]

Accuracy score: 0.8164251207729468
Precision score: 0.7333333333333333
Recall score: 0.13253012048192772
f1 score: 0.22448979591836735
